In [20]:
import sqlite3
import pandas as pd

# Tạo kết nối cơ sở dữ liệu SQLite trong bộ nhớ
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Khởi tạo bảng và chèn dữ liệu
cursor.executescript('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);

CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

INSERT INTO course (id, course_name) VALUES
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc');

INSERT INTO student (student_id, name, class, course_id, score) VALUES
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', NULL, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc', 'Kinh Te', NULL, 7.2),
(10, 'Cao Thi Hanh', 'May Tinh', NULL, 7.0);
''')
conn.commit()

# Kiểm tra dữ liệu
display(pd.read_sql_query('SELECT * FROM student;', conn))
display(pd.read_sql_query('SELECT * FROM course;', conn))

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


## Part 1

In [22]:
# Tích Decartes
df_cross = pd.read_sql_query('SELECT * FROM student CROSS JOIN course;', conn)

# INNER JOIN
df_inner = pd.read_sql_query('SELECT * FROM student INNER JOIN course ON student.course_id = course.id;', conn)

# LEFT JOIN
df_left = pd.read_sql_query('SELECT * FROM student LEFT JOIN course ON student.course_id = course.id;', conn)

# RIGHT JOIN & FULL OUTER JOIN 
df_right = pd.read_sql_query('SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id;', conn)
df_full = pd.read_sql_query('SELECT * FROM student FULL OUTER JOIN course ON student.course_id = course.id;', conn)

# Hiển thị thử kết quả của LEFT JOIN
display(df_left)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


### Kết luận
Tùy vào mục đích sử dụng mà lựa chọn phép JOIN phù hợp. Trong các bài toán phân tích dữ liệu thực tế,

INNER JOIN thường được ưu tiên vì đảm bảo tính chính xác và nhất quán của dữ liệu.

Trong khi đó, LEFT JOIN, RIGHT JOIN và FULL OUTER JOIN đóng vai trò quan trọng trong việc kiểm tra, phát hiện và xử lý dữ liệu thiếu hoặc không hợp lệ. 

Riêng CROSS JOIN ít được sử dụng trong phân tích thực tế do tạo ra dữ liệu dư thừa, nhưng vẫn có ý nghĩa trong các bài toán sinh tổ hợp.

## Part 2

In [23]:
# Làm sạch dữ liệu: Điền mã môn học 26 (Tin học) cho các sinh viên thiếu dữ liệu và xóa môn không hợp lệ
cursor.executescript('''
UPDATE student SET course_id = 26 WHERE course_id IS NULL;
DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course);
''')
conn.commit()

# a. Thống kê theo lớp học
df_class_stats = pd.read_sql_query('''
SELECT class, COUNT(student_id) AS total_students, AVG(score) AS average_score
FROM student
GROUP BY class;
''', conn)
display(df_class_stats)

# b. Thống kê theo môn học
df_course_stats = pd.read_sql_query('''
SELECT course_id, COUNT(student_id) AS total_students, AVG(score) AS average_score
FROM student
GROUP BY course_id;
''', conn)
display(df_course_stats)

# c. Phân loại thi đua dựa trên điểm trung bình của từng môn
df_classification = pd.read_sql_query('''
SELECT course_id, AVG(score) AS average_score,
CASE
    WHEN AVG(score) >= 9.0 THEN 'Xuất sắc'
    WHEN AVG(score) >= 6.0 AND AVG(score) <= 8.9 THEN 'Tốt'
    ELSE 'Kém'
END AS classification
FROM student
GROUP BY course_id;
''', conn)
display(df_classification)

,class,total_students,average_score
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000


,course_id,total_students,average_score
0,12,1,6.700000
1,26,3,7.366667
2,34,2,9.200000


,course_id,average_score,classification
0,12,6.700000,Tốt
1,26,7.366667,Tốt
2,34,9.200000,Xuất sắc


## Part 3

In [24]:
# a. Xếp hạng theo điểm số
df_rank_score = pd.read_sql_query('''
SELECT student_id, name, score, RANK() OVER(ORDER BY score DESC) AS rank_overall
FROM student;
''', conn)

# b. Xếp hạng theo lớp học
df_rank_class = pd.read_sql_query('''
SELECT student_id, name, class, score, RANK() OVER(PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student;
''', conn)

# c. Xếp hạng theo môn học
df_rank_course = pd.read_sql_query('''
SELECT student_id, name, course_id, score, RANK() OVER(PARTITION BY course_id ORDER BY score DESC) AS rank_course
FROM student;
''', conn)

# Lấy Top 3 và Bottom 3 từ bảng xếp hạng chung làm ví dụ
top_3 = df_rank_score.head(3)
bottom_3 = df_rank_score.tail(3)

display(top_3)
display(bottom_3)

,student_id,name,score,rank_overall
0,2,Tran Thi Lan,9.2,1
1,7,Bui Tien Dung,9.2,1
2,3,Pham Van Nam,7.9,3


,student_id,name,score,rank_overall
3,9,Duong Huu Phuc,7.2,4
4,10,Cao Thi Hanh,7.0,5
5,1,Nguyen Minh Hoang,6.7,6


## Part 4

In [25]:
# Thêm cột graduation_date và cập nhật dựa trên thứ hạng
cursor.executescript('''
ALTER TABLE student ADD COLUMN graduation_date DATETIME;

WITH RankedStudents AS (
    SELECT student_id, RANK() OVER(ORDER BY score DESC) AS rank_val
    FROM student
)
UPDATE student
SET graduation_date = DATE('now', '+' || (
    SELECT rank_val FROM RankedStudents WHERE RankedStudents.student_id = student.student_id
) || ' days');
''')
conn.commit()

# Xem kết quả tổng hợp cuối cùng
df_final = pd.read_sql_query('SELECT * FROM student;', conn)
display(df_final)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-09
1,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-04
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-06
3,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-04
4,9,Duong Huu Phuc,Kinh Te,26,7.2,2026-04-07
5,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-08
